# Dataset Info

Overview of the YC dataset: what we have, how fresh it is, and what it looks like.

In [ ]:
import pandas as pd
import sqlite3
import os
from datetime import datetime

DB_PATH = "../data/yc_collaboration.db"
CSV_PATH = "../data/yc_collaboration_companies.csv"

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql("SELECT * FROM companies", conn)
conn.close()

print("Dataset loaded.")

---
## 1. Data Source & Freshness

In [ ]:
db_modified = datetime.fromtimestamp(os.path.getmtime(DB_PATH)).strftime("%Y-%m-%d %H:%M")
csv_modified = datetime.fromtimestamp(os.path.getmtime(CSV_PATH)).strftime("%Y-%m-%d %H:%M")
db_size = os.path.getsize(DB_PATH) / (1024 * 1024)
csv_size = os.path.getsize(CSV_PATH) / (1024 * 1024)

info = {
    "Source": "yc-oss.github.io/api (community-maintained, from YC Algolia index)",
    "Source last updated": "2026-02-08",
    "Our scrape date": db_modified,
    "SQLite file": f"{DB_PATH} ({db_size:.2f} MB)",
    "CSV file": f"{CSV_PATH} ({csv_size:.2f} MB)",
    "Total companies": len(df),
}

for k, v in info.items():
    print(f"{k:<25} {v}")

---
## 2. Schema & Columns

In [ ]:
col_info = pd.DataFrame({
    "Column": df.columns,
    "Type": df.dtypes.values,
    "Non-Null": df.notnull().sum().values,
    "Null": df.isnull().sum().values,
    "Fill %": (df.notnull().sum().values / len(df) * 100).round(1),
    "Unique": df.nunique().values,
    "Sample": [df[col].dropna().iloc[0] if df[col].notnull().any() else "" for col in df.columns],
})

# Truncate sample values for display
col_info["Sample"] = col_info["Sample"].astype(str).str[:60]

print(f"Columns: {len(df.columns)}")
print(f"Rows: {len(df):,}")
print()
col_info

---
## 3. Status Breakdown

In [ ]:
status = df["status"].value_counts().reset_index()
status.columns = ["Status", "Count"]
status["Percentage"] = (status["Count"] / status["Count"].sum() * 100).round(1).astype(str) + "%"
status

---
## 4. Batch Range & Distribution

In [ ]:
df["batch_year"] = df["batch"].str.extract(r"(\d{4})").astype(float)

batches = df["batch"].value_counts().sort_index()
years = df["batch_year"].dropna()

print(f"{'Oldest batch':<25} {batches.index[0]} ({int(batches.iloc[0])} companies)")
print(f"{'Newest batch':<25} {batches.index[-1]} ({int(batches.iloc[-1])} companies)")
print(f"{'Total batches':<25} {len(batches)}")
print(f"{'Year range':<25} {int(years.min())} - {int(years.max())} ({int(years.max() - years.min())} years)")
print()

# Top 5 largest batches
print("=== Largest Batches ===")
for batch, count in batches.sort_values(ascending=False).head(5).items():
    print(f"  {batch:<20} {count} companies")

In [ ]:
# Companies per year
yearly = df.groupby("batch_year").size()

print("=== Companies per Year ===")
for year, count in yearly.items():
    bar = "|" * int(count / 10)
    print(f"  {int(year)}  {count:>4}  {bar}")

---
## 5. Industry Breakdown

In [ ]:
industries = df["industry"].value_counts().reset_index()
industries.columns = ["Industry", "Count"]
industries["Percentage"] = (industries["Count"] / industries["Count"].sum() * 100).round(1).astype(str) + "%"
industries

---
## 6. Stage Breakdown

In [ ]:
stage = df["stage"].value_counts().reset_index()
stage.columns = ["Stage", "Count"]
stage["Percentage"] = (stage["Count"] / stage["Count"].sum() * 100).round(1).astype(str) + "%"
stage

---
## 7. Hiring Overview

In [ ]:
total = len(df)
hiring = int(df["is_hiring"].sum())
active_hiring = int(df[(df["is_hiring"] == 1) & (df["status"] == "Active")].shape[0])

print(f"{'Total companies':<30} {total:,}")
print(f"{'Currently hiring':<30} {hiring:,} ({hiring/total*100:.1f}%)")
print(f"{'Active & hiring':<30} {active_hiring:,} ({active_hiring/total*100:.1f}%)")

---
## 8. Team Size Stats

In [ ]:
ts = df["team_size"].describe()
print(f"{'Min':<20} {int(ts['min'])}")
print(f"{'25th percentile':<20} {int(ts['25%'])}")
print(f"{'Median':<20} {int(ts['50%'])}")
print(f"{'Mean':<20} {int(ts['mean'])}")
print(f"{'75th percentile':<20} {int(ts['75%'])}")
print(f"{'Max':<20} {int(ts['max'])}")
print(f"{'Missing':<20} {df['team_size'].isnull().sum()}")

---
## 9. Geography

In [ ]:
regions = df["regions"].str.split("; ").explode().str.strip()
regions = regions[regions != ""]

print("=== Top Regions ===")
for region, count in regions.value_counts().head(15).items():
    print(f"  {region:<40} {count}")

print()

cities = df["location"].str.split(",").str[0].str.strip()
print("=== Top Cities ===")
for city, count in cities.value_counts().head(15).items():
    print(f"  {city:<40} {count}")

---
## 10. Tag Coverage

In [ ]:
all_tags = df["tags"].str.split("; ").explode().str.strip()
all_tags = all_tags[(all_tags != "") & all_tags.notnull()]

no_tags = (df["tags"].isna() | (df["tags"] == "")).sum()

print(f"{'Unique tags':<25} {all_tags.nunique()}")
print(f"{'Companies with tags':<25} {len(df) - no_tags} ({(len(df) - no_tags)/len(df)*100:.1f}%)")
print(f"{'Companies without tags':<25} {no_tags} ({no_tags/len(df)*100:.1f}%)")
print(f"{'Avg tags per company':<25} {all_tags.groupby(level=0).count().mean():.1f}")

---
## 11. Sample Records

In [ ]:
print("=== 5 Random Active Companies ===")
df[df["status"] == "Active"].sample(5, random_state=42)[["name", "one_liner", "batch", "team_size", "industry", "location"]]

In [ ]:
print("=== 5 Random Hiring Companies ===")
df[df["is_hiring"] == 1].sample(5, random_state=42)[["name", "one_liner", "website", "batch", "team_size", "location"]]